In [31]:
import pandas as pd

asv = pd.read_csv(r"../data/ibd_asv_table.txt", sep= '\t', index_col=0)
meta = pd.read_csv(r"../data/ibd_meta.csv", index_col=0)
taxa = pd.read_csv(r"../data/ibd_taxa.txt", sep= '\t', index_col=0)

print(asv.shape)
print(meta.shape)
print(taxa.shape)

(1681, 43)
(43, 1)
(1681, 7)


In [32]:
print("ASV table:")
print(asv.shape)
print(asv.iloc[:3, :5]) 
print()
print("Metadata:")
print(meta.head())
print()
print("Taxa:")
print(taxa.head())

ASV table:
(1681, 43)
                                                    206700  206701  206702  \
#NAME                                                                        
TACGGAGGATCCGAGCGTTATCCGGATTTATTGGGTTTAAAGGGAGC...      18      53      55   
AACGTAGGTCACAAGCGTTGTCCGGAATTACTGGGTGTAAAGGGAGC...      10       0       0   
TACGGAGGGTGCAAGCGTTAATCGGAATTACTGGGCGTAAAGCGCAC...   12315   13605   30872   

                                                    206703  206704  
#NAME                                                               
TACGGAGGATCCGAGCGTTATCCGGATTTATTGGGTTTAAAGGGAGC...    8638    4154  
AACGTAGGTCACAAGCGTTGTCCGGAATTACTGGGTGTAAAGGGAGC...     979    1675  
TACGGAGGGTGCAAGCGTTAATCGGAATTACTGGGCGTAAAGCGCAC...     275     366  

Metadata:
          Class
#NAME          
206700       CD
206701       CD
206702       CD
206703  Control
206704  Control

Taxa:
                                                        Kingdom  \
#TAXONOMY                                    

In [33]:
print(meta['Class'].value_counts())
print()
meta.index = meta.index.astype(str) #convert to string for matching

#query sample ID match
asv_samples = set(asv.columns)
meta_samples = set(meta.index)
print(f"Samples in ASV table: {len(asv_samples)}")
print(f"Samples in Metadata: {len(meta_samples)}")
print (f"Matching: {len(asv_samples & meta_samples)}")
print (f"Only in ASV: {len(asv_samples - meta_samples)}")
print (f"Only in Metadata: {len(meta_samples - asv_samples)}")


Class
CD         23
Control    20
Name: count, dtype: int64

Samples in ASV table: 43
Samples in Metadata: 43
Matching: 43
Only in ASV: 0
Only in Metadata: 0


In [34]:
depth = asv.sum(axis=0).sort_values()
print(depth.head(10))

219659     3857
206768     9545
206701    20914
219656    21765
206767    25803
219634    26089
206718    28376
219633    28793
219691    30937
219675    32485
dtype: int64


In [35]:
# drop two lowest depth samples
samples_to_drop = ['219659', '206768']
asv = asv.drop(columns=samples_to_drop)
meta = meta.drop(index=samples_to_drop)

print(f"ASV table after dropping low-depth samples: {asv.shape}")
print(f"Metadata after dropping low-depth samples: {meta.shape}")
print()
print(meta['Class'].value_counts())

ASV table after dropping low-depth samples: (1681, 41)
Metadata after dropping low-depth samples: (41, 1)

Class
CD         21
Control    20
Name: count, dtype: int64


In [36]:
#remove singletons
total_counts = asv.sum(axis=1)
print(f"ASVs before singleton removal: {asv.shape[0]}")
print(f"Singleton ASVs: (total count = 1): {(total_counts == 1).sum()}")

asv = asv[total_counts > 1]
print(f"ASVs after singleton removal: {asv.shape[0]}")

ASVs before singleton removal: 1681
Singleton ASVs: (total count = 1): 0
ASVs after singleton removal: 1498


In [37]:
#prevalence filter: ASV must have >= 4 reads in 20% of samples
import numpy as np
min_count = 4
min_prevalence = 0.2
min_samples = int(np.ceil(min_prevalence * asv.shape[1]))
print(f"Minimum samples threshold: {min_samples}")

#count number of samples each ASV meets the count threshold
prevalence = (asv >= min_count).sum(axis=1)

#keep ASVs that meet the prevalence threshold
asv = asv[prevalence >= min_samples]
print(f"ASVs after prevalence filtering: {asv.shape[0]}")

Minimum samples threshold: 9
ASVs after prevalence filtering: 144


In [38]:
#IQR variance filter - remove the bottom 10% least variable ASVs
variance = asv.var(axis=1)
iqr_threshold = variance.quantile(0.1)
print(f"IQR variance threshold: {iqr_threshold:.4f}")

asv = asv[variance > iqr_threshold]
print(f"ASVs after IQR variance filtering: {asv.shape[0]}")

IQR variance threshold: 412.8643
ASVs after IQR variance filtering: 129


In [39]:
from scipy.stats import gmean
import numpy as np

# CLR transformation
# add pseudocount of 1 to avoid log(0)
asv_pseudo = asv + 1

# calculate geometric mean per sample (axis=0 = per column)
geo_means = asv_pseudo.apply(gmean, axis=0)

# divide each value by the sample geometric mean, then log transform
asv_clr = np.log(asv_pseudo.divide(geo_means, axis=1))

print(f"CLR transformed table shape: {asv_clr.shape}")
print(f"Value range: {asv_clr.values.min():.3f} to {asv_clr.values.max():.3f}")
print(asv_clr.iloc[:3, :3])

CLR transformed table shape: (129, 41)
Value range: -3.563 to 10.123
                                                      206700    206701  \
#NAME                                                                    
TACGGAGGATCCGAGCGTTATCCGGATTTATTGGGTTTAAAGGGAGC...  1.645797  3.703864   
AACGTAGGTCACAAGCGTTGTCCGGAATTACTGGGTGTAAAGGGAGC...  1.099253 -0.285120   
TACGGAGGGTGCAAGCGTTAATCGGAATTACTGGGCGTAAAGCGCAC...  8.120012  9.233146   

                                                       206702  
#NAME                                                          
TACGGAGGATCCGAGCGTTATCCGGATTTATTGGGTTTAAAGGGAGC...   3.810760  
AACGTAGGTCACAAGCGTTGTCCGGAATTACTGGGTGTAAAGGGAGC...  -0.214592  
TACGGAGGGTGCAAGCGTTAATCGGAATTACTGGGCGTAAAGCGCAC...  10.123046  


In [41]:
print(f"CLR table: {asv_clr.shape}")
print(f"Metadata: {meta.shape}")
print(f"Sample IDs match: {set(asv_clr.columns) == set(meta.index)}")

CLR table: (129, 41)
Metadata: (41, 1)
Sample IDs match: True
